# Analyical Solution of 1D Kitaev Chain Hamiltonian

In a 1D Kitaev chain, the chemical potential $\mu$ is the primary tuning parameter that drives the topological phase transition. It controls three related but distinct observables, each of which shows up directly in the PINN surrogate pipeline:

- the **energy spectrum**, $E(\mu)$ — the quantity labelled `E` in `sweep_lowest_nonnegative_state`;
- the **probability density profile**, $\rho(n) = |\psi_n|^2$ along the chain — `particle_prob` / `hole_prob`;
- the **edge weight**, the integral of $\rho(n)$ over a window of sites at the boundaries — `particle_edge_weight` / `hole_edge_weight` / `combined_edge_weight`.

All three depend on the ratio between $\mu$, the hopping amplitude $t$, and the pairing amplitude $\Delta$ (assumed real and positive here). $t$ sets the overall energy scale; $\mu/t$ and $\Delta/t$ together set the shape of all three observables.

---

## The Mechanism

Away from special points, the Kitaev chain is not exactly solvable, so the spectrum and wavefunctions below are described qualitatively, with exact results given only where they exist in closed form. The bulk dispersion (infinite chain, periodic boundary conditions) is

$$
E(k) = \pm\sqrt{\left(\mu + 2t\cos k\right)^2 + 4\Delta^2\sin^2 k}\, ,
$$

which is useful for locating the phase boundary: the bulk gap closes ($E=0$ for some $k$) exactly when $\mu = \pm 2t$. This is the origin of the $|\mu| = 2t$ transition used throughout.

For a finite open chain in the topological phase, two Majorana operators localised at opposite ends hybridise with an amplitude that decays exponentially with chain length $L$, splitting the two zero modes to a small but finite energy $E \sim e^{-L/\xi}$, where $\xi$ is the Majorana localisation length. $\xi$ is set by $\mu$, $t$, and $\Delta$ together, and it is the single quantity that governs the shape of all three observables below.

---

## 1. The Exactly Solvable Point ($\mu = 0$, $\Delta = t$)

- **Behaviour**: at this specific point — not $\mu=0$ alone — the Kitaev chain decouples exactly into a dimerised chain of Majorana operators, each site's two Majorana components pairing entirely with a neighbour on one side and not at all with the other.
- **Energy spectrum**: the two edge modes are exactly degenerate at $E=0$ for any chain length $L \geq 2$, with no finite-size correction — not just approximately small, but exactly zero within the model.
- **Probability density**: $\rho(n)$ is a delta function on the outermost site at each end; there is no decay to characterise because there is no leakage at all.
- **Edge weight**: exactly $1$ at each end, $0$ everywhere else.

**This is worth flagging explicitly**: $\mu=0$ alone is not sufficient for this exact result — it also requires $\Delta = t$. At $\mu=0$ with $\Delta \neq t$ (e.g. the toy-model defaults used in this project, $t=1$, $\Delta=0.5$), the chain still decouples into two independent Majorana sub-chains, but with couplings $(t+\Delta)$ and $(t-\Delta)$ rather than $(2t, 0)$. The localisation length at $\mu=0$ is then

$$
\xi(\mu{=}0) = \frac{1}{\ln\left(\dfrac{t+\Delta}{|t-\Delta|}\right)}\, ,
$$

which is finite and only vanishes as $\Delta \to t$. In other words, in this project's own toy-model parameterisation ($\Delta = 0.5t$), $\mu=0$ still gives the *shortest* localisation length in the domain, but the edge mode is not strictly single-site — a detail worth keeping in mind when interpreting `combined_edge_weight` at the centre of the sweep; it should approach a high value close to 1, but not exactly 1, and the residual bulk leakage should be visible in `particle_prob`/`hole_prob` as a small non-zero tail away from the boundary sites.

---

## 2. Bulk Leakage: The Interior of the Topological Phase ($0 < |\mu| < 2t$)

- **Behaviour**: as $|\mu|$ increases away from its value at the sweet spot but stays below $2t$, the system remains topological, but the localisation length grows.
- **Energy spectrum**: the finite-size splitting grows smoothly away from zero,
$$
E(\mu) \sim e^{-L/\xi(\mu)}\, ,
$$
  generally with an oscillatory prefactor $\cos(k_F L + \varphi)$ when $\Delta \neq t$, since the decaying mode has a complex wavevector with both a real (oscillatory) and imaginary (decaying) part. In practice this means $E(\mu)$ need not increase perfectly monotonically with $|\mu|$ — small non-monotonic wiggles superimposed on the overall exponential growth are expected finite-size physics, not necessarily a symptom of a training or numerical problem when this shows up in the PINN's predicted spectrum.
- **Probability density**: $\rho(n)$ decays exponentially away from each edge into the bulk,
$$
\rho(n) \sim e^{-2n/\xi(\mu)}\, ,
$$
  again with possible oscillatory modulation along $n$ for $\Delta \neq t$. The two edge modes' density profiles start to overlap in the centre of a finite chain as $\xi$ grows, which is the microscopic origin of the level splitting above.
- **Edge weight**: decreases monotonically from its value near the sweet spot as $|\mu|$ grows, since more of $\rho(n)$'s weight has migrated from the boundary window into the interior.

---

## 3. Critical Point: Gap Closing ($|\mu| = 2t$)

- **Behaviour**: the bulk gap closes at $k=0$ (for $\mu = -2t$) or $k=\pi$ (for $\mu=+2t$).
- **Energy spectrum**: $\xi \to \infty$, so the exponential-splitting picture above breaks down entirely — the lowest-lying "edge" state is no longer exponentially close to zero by virtue of being edge-localised; it merges continuously with the bulk continuum of near-zero bulk states. $E(\mu)$ typically shows its steepest, most sensitive region here — the hardest part of the domain for any surrogate (neural or otherwise) to resolve precisely, and the point directly implicated in the Attempt 5 phase-boundary residual spikes discussed in the Stage 1 findings.
- **Probability density**: $\rho(n)$ is fully delocalised — no exponential envelope remains, and the profile looks like a generic extended bulk state rather than two decaying edge lobes.
- **Edge weight**: falls to whatever value a generic delocalised bulk state happens to have in the boundary window — small, and no longer physically meaningful as an "edge" quantity, since there is no longer a well-defined edge mode to distinguish from the bulk.

---

## 4. The Trivial Phase ($|\mu| > 2t$)

- **Behaviour**: the bulk gap reopens, but the system is now topologically trivial — the Majorana operators recombine into ordinary, non-topological fermionic excitations.
- **Energy spectrum**: $E(\mu)$ grows roughly linearly with $|\mu|$ far from the transition, approaching the chemical-potential-dominated limit $E \to |\mu| - 2t$ for $|\mu| \gg 2t$; there is no longer a pinned near-zero state at all — the "lowest non-negative eigenvalue" tracked by `sweep_lowest_nonnegative_state` here is just the smallest ordinary bulk excitation energy, not an edge mode.
- **Probability density**: $\rho(n)$ is extended across the full chain (a generic bulk eigenstate), possibly modulated by the tight-binding standing-wave pattern set by $\mu/t$, but with no exponential edge envelope.
- **Edge weight**: settles to a small, $L$-dependent baseline value consistent with an extended state's average density over the edge window — order $\mathcal{O}(n_{\text{edge}}/L)$ rather than order $1$.

---

## Phase Summary

| Chemical potential | Topological phase? | Energy spectrum $E(\mu)$ | Probability density $\rho(n)$ | Edge weight |
|---|---|---|---|---|
| $\mu = 0$, $\Delta = t$ (exact point) | Yes (ideal) | Exactly $0$, no finite-size correction | Delta function on outermost site | Exactly $1$ at each end |
| $\mu = 0$, $\Delta \neq t$ | Yes | Smallest non-zero splitting in the domain | Shortest decay length in the domain, small residual bulk tail | Near-maximal, not exactly $1$ |
| $0 < \lvert\mu\rvert < 2t$ | Yes | Exponentially small, $\sim e^{-L/\xi(\mu)}$, possibly oscillatory | Exponential decay from each edge, length $\xi(\mu)$ | Decreasing monotonically toward the transition |
| $\lvert\mu\rvert = 2t$ | Phase transition | Steepest, most sensitive region; edge state merges with bulk continuum | Fully delocalised, no exponential envelope | Falls to a non-meaningful bulk-like value |
| $\lvert\mu\rvert > 2t$ | No (trivial) | Grows roughly linearly, $E \to \lvert\mu\rvert - 2t$ far from transition | Extended bulk state across the whole chain | Small, $\mathcal{O}(n_{\text{edge}}/L)$ baseline |

---


## Import Packages

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from ipywidgets import IntSlider, interact
from sesh import Session

from kitaev.analytical import KitaevChainHamiltonian

sns.set_style("whitegrid")

## Initialise Session

In [3]:
session = Session(
    name="1d-kitaev-chain-analytical",
    output_root=Path("../results/logs"),
    enable_mlflow=False,
)

2026-08-25 17:33:17 | INFO | 1d-kitaev-chain-analytical | Experiment session: 1d-kitaev-chain-analytical initialised.
2026-08-25 17:33:17 | INFO | 1d-kitaev-chain-analytical | Seeding engine completed using base reference: 42
2026-08-25 17:33:17 | INFO | 1d-kitaev-chain-analytical | Local workflow runtime tracking directed to: ../results/logs/20260825_173317_1d-kitaev-chain-analytical


## Define 1D Kitaev Hamiltonian

In [4]:
N = 20
session.info(f"Number of sites, N = {N}")
t = 1
session.info(f"Hopping amplitude, t = {t}")

2026-08-25 17:33:18 | INFO | 1d-kitaev-chain-analytical | Number of sites, N = 20
2026-08-25 17:33:18 | INFO | 1d-kitaev-chain-analytical | Hopping amplitude, t = 1


In [9]:
H = KitaevChainHamiltonian(n_sites=N, hopping=t)

example = H.build(mu=0.5)
session.info("Example Hamiltonian:\n" + str(example))
session.info(f"Hamiltonian Shape: {example.shape}")

2026-08-25 21:07:11 | INFO | 1d-kitaev-chain-analytical | Example Hamiltonian:
[[-0.5 -1.   0.  ...  0.   0.   0. ]
 [-1.  -0.5 -1.  ...  0.   0.   0. ]
 [ 0.  -1.  -0.5 ...  0.   0.   0. ]
 ...
 [ 0.   0.   0.  ...  0.5  1.   0. ]
 [ 0.   0.   0.  ...  1.   0.5  1. ]
 [ 0.   0.   0.  ...  0.   1.   0.5]]
2026-08-25 21:07:11 | INFO | 1d-kitaev-chain-analytical | Hamiltonian Shape: (40, 40)


## Diagonlise Hamiltonian and Solve for Eigenvalues and Eigenvectors

In [ ]:
mu_array = np.linspace(-3, 3, 300)

n_mu = len(mu_array)
n_edge_sites = 2  # number of edge sites counted in edge weight!
split_index = N  # lowest non-negative eigenvalue/eigenvector
transition = 2 * H.hopping  # |mu| = 2t topological phase boundary

E = np.zeros(n_mu)
particle_prob = np.zeros((n_mu, N))
hole_prob = np.zeros((n_mu, N))
particle_edge_weight = np.zeros(n_mu)
hole_edge_weight = np.zeros(n_mu)
combined_edge_weight = np.zeros(n_mu)

edge_sites = np.concatenate(
    [
        np.arange(n_edge_sites),
        np.arange(N - n_edge_sites, N),
    ]
)

for i, mu in enumerate(mu_array):
    eigenvalues, eigenvectors = np.linalg.eigh(H.build(mu))
    order = np.argsort(np.abs(eigenvalues))
    E[i] = np.abs(eigenvalues[order[0]])

    if abs(mu) < transition:
        # Topological phase: the +-lambda_1 pair is (near-)degenerate, so a
        # single eigh column is an arbitrary member of the doublet and
        # routinely comes out localised on one edge once the splitting
        # drops below machine precision. Use the gauge-invariant pair
        # density rho/2 -- the density a balanced edge eigenstate carries,
        # symmetric under n -> N-1-n by construction.
        near = eigenvectors[:, order[:2]]
        particle_prob[i] = (np.abs(near[:N, :]) ** 2).sum(axis=1) / 2
        hole_prob[i] = (np.abs(near[N:, :]) ** 2).sum(axis=1) / 2
    else:
        # Trivial phase: no degeneracy, the single lowest-|E| eigenvector
        # is well defined.
        psi = eigenvectors[:, split_index]
        particle_prob[i] = np.abs(psi[:N]) ** 2
        hole_prob[i] = np.abs(psi[N:]) ** 2

    particle_edge_weight[i] = particle_prob[i, edge_sites].sum()
    hole_edge_weight[i] = hole_prob[i, edge_sites].sum()
    combined_edge_weight[i] = particle_edge_weight[i] + hole_edge_weight[i]

## Visualisations

### Lowest Eigen-Energy, $E$ vs Chemical Potential, $\mu$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

sns.lineplot(x=mu_array, y=E, linestyle="-", color="black")

# --- Add Topological Transition Lines ---
ax.axvline(
    x=-transition,
    color="r",
    linestyle="--",
    linewidth=1.5,
    label=r"Transition ($\mu = -2t$)",
)
ax.axvline(
    x=transition,
    color="r",
    linestyle="--",
    linewidth=1.5,
    label=r"Transition ($\mu = 2t$)",
)

# Add a legend to identify the transition lines
ax.legend(frameon=True, facecolor="white", edgecolor="none")

ax.set_title(r"$E$ vs $\mu$")  # Add a title
ax.set_xlabel(r"Chemical Potential, $\mu$")  # X-axis label
ax.set_ylabel(r"Energy, E")  # Y-axis label
ax.grid(True)  # Add grid lines

plt.savefig(session.path() / "combined_edge_weight_vs_mu.png", dpi=300)

plt.show()

## $|\psi_p|^2$ and $|\psi_h|^2$ Evolution

In [ ]:
%matplotlib widget
sites = np.arange(N)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax_p, ax_h = axes


def plot_probability_densities(i: int):

    ax_p.cla()
    ax_h.cla()

    sns.lineplot(x=sites, y=particle_prob[i], linestyle="-", color="blue", ax=ax_p)
    ax_p.set_title("Particle (electron) sector")
    sns.lineplot(x=sites, y=hole_prob[i], linestyle="-", color="red", ax=ax_h)
    ax_h.set_title("Hole sector")

    ymax = max(particle_prob.max(), hole_prob.max()) * 1.1
    for ax in (ax_p, ax_h):
        ax.set_xlabel("Site index n")
        ax.set_ylim(0, ymax)
        ax.grid(alpha=0.3)

    ax_p.set_ylabel(r"$|\psi_n|^2$")

    regime = "Topological" if abs(mu_array[i]) < transition else "Trivial"
    note = r" ($\rho/2$)" if abs(mu_array[i]) < transition else ""
    fig.suptitle(
        f"{regime} Regime{note}\n" + r"$\mu$ =" + f"{mu_array[i]:.2f}, E = {E[i]:.4f}"
    )
    fig.tight_layout()
    fig.canvas.draw_idle()


interact(
    plot_probability_densities,
    i=IntSlider(min=0, max=len(mu_array) - 1, step=1, value=149, description="i"),
);

### Combined Edge Weight vs Chemical Potential, $\mu$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Use seaborn's style for better aesthetics
sns.lineplot(x=mu_array, y=combined_edge_weight, linestyle="-", color="black")

# --- Add Topological Transition Lines ---
ax.axvline(
    x=-transition,
    color="r",
    linestyle="--",
    linewidth=1.5,
    label=r"Transition ($\mu = -2t$)",
)
ax.axvline(
    x=transition,
    color="r",
    linestyle="--",
    linewidth=1.5,
    label=r"Transition ($\mu = 2t$)",
)

# Add a legend to identify the transition lines
ax.legend(frameon=True, facecolor="white", edgecolor="none")

ax.set_title(r"Combined Edge Weight (%) vs $\mu$")  # Add a title
ax.set_xlabel(r"Chemical Potential, $\mu$")  # X-axis label
ax.set_ylabel("Combined Edge Weight (%)")  # Y-axis label
ax.grid(True)  # Add grid lines

plt.savefig(session.path() / "combined_edge_weight_vs_mu.png", dpi=300)

plt.show()